In [11]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

df = pd.read_csv('../data/raw/pre-processed.csv') 

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

X = df['preprocessed_news'].astype(str).values
y = df['label'].values
y = np.where(y == 'fake', 1, 0)


X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Treino: {len(X_train)} amostras")
print(f"Validação: {len(X_val)} amostras")
print(f"Teste: {len(X_test)} amostras")

max_words = 10000  
max_len = 300      

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

os.makedirs('../data/processed', exist_ok=True)

np.save('../data/processed/X_train.npy', X_train_pad)
np.save('../data/processed/X_val.npy', X_val_pad)
np.save('../data/processed/X_test.npy', X_test_pad)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_val.npy', y_val)
np.save('../data/processed/y_test.npy', y_test)

print("\nOs 6 arquivos foram salvos com sucesso na pasta 'data/processed/'!")

Treino: 5040 amostras
Validação: 1080 amostras
Teste: 1080 amostras

Os 6 arquivos foram salvos com sucesso na pasta 'data/processed/'!


In [12]:
import numpy as np

X_train = np.load('../data/processed/X_train.npy')
y_train = np.load('../data/processed/y_train.npy')

print("TESTE DE INTEGRIDADE DOS DADOS")
print(f"Formato do X_train (Textos convertidos): {X_train.shape}")
print(f"Formato do y_train (Rótulos numéricos): {y_train.shape}")

print("\nExemplo da primeira notícia codificada em números:")
print(X_train[0])

print(f"\nClasse dessa primeira notícia (1 para Fake, 0 para True): {y_train[0]}")

TESTE DE INTEGRIDADE DOS DADOS
Formato do X_train (Textos convertidos): (5040, 300)
Formato do y_train (Rótulos numéricos): (5040,)

Exemplo da primeira notícia codificada em números:
[   1 4842    1 4592 8215  256 4908   99  780  104 4974   79  913  895
  992 4842 4103    1 8216   44    1 6337  157 1227  454    1 2557  718
 1637  521 4975 4842    1 8003  104 4974    1  220   99  780  887    1
   15   58   71  521 4103  265 8883   20 1969   49  729   71  220   99
  780    2  279   22  100 3460  183 2832 1267 7804  157  454 4103    1
  894  318 3678    1 4103 1098    1    1  639  822  454 5446 9405  247
 8004    1  713    1 6337   13   44  760   91    1  521 1414 4103 1297
  558 1861 2872 9699 2833    5   41    1   15   58  745  104 4974 1305
  625 3878    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0

In [13]:
import pickle
import os

os.makedirs('../models', exist_ok=True)

with open('../models/tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Tokenizer SALVO com sucesso em 'models/tokenizer.pickle'!")

Tokenizer SALVO com sucesso em 'models/tokenizer.pickle'!
